In [58]:
import pandas as pd
df = pd.read_csv("SMSSpamCollection.txt",
                sep ="\t",
                header =None,
                names = ['label', 'messages'])
df.head()

,label,messages
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [59]:
print("Shape of df",df.shape)

Shape of df (5572, 2)


In [60]:
print("\nClass counts:")
print(df["label"].value_counts())


Class counts:
label
ham     4825
spam     747
Name: count, dtype: int64


In [61]:
# The data is IMBALANCED: far more ham than spam. This is why accuracy alone
# is misleading. A model that always says "ham" would already score ~87%.
# That's why we also check precision, recall and F1 later.

In [62]:
#converts text labels into numbers spam --> 1 and ham --> 0
df['label'] = df['label'].map({"spam":1,"ham":0})
print(df.head())

   label                                           messages
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


## Text processing with nltk

In [63]:
nltk.download("punkt")       # tokenizer data
nltk.download("punkt_tab")   # needed by newer NLTK versions
nltk.download("stopwords") 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\navee\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [64]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [65]:
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

In [66]:
def preprocess_text(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    stemmed = [stemmer.stem(word) for word in tokens]
    return " ".join(stemmed)

In [67]:
preprocess_text("My name is naveen. i liked to run on the streets")

'name naveen . like run street'

In [68]:
#apply on df

df['cleaned_messages'] = df['messages'].apply(preprocess_text)
df.head()

,label,messages,cleaned_messages
0,0,"Go until jurong point, crazy.. Available only ...","go jurong point , crazi .. avail bugi n great ..."
1,0,Ok lar... Joking wif u oni...,ok lar ... joke wif u oni ...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor ... u c alreadi say ...
4,0,"Nah I don't think he goes to usf, he lives aro...","nah n't think goe usf , live around though"


In [69]:
#model training
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

In [70]:
X = df['cleaned_messages']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

## # --- Method A: Bag of Words (word counts) ---

In [71]:
bow_vectoriser = CountVectorizer()
X_train_bow = bow_vectoriser.fit_transform(X_train)
X_test_bow = bow_vectoriser.transform(X_test)
print(X_train_bow[1])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (1, 6540)>
  Coords	Values
  (0, 4490)	2
  (0, 1733)	1
  (0, 3144)	1
  (0, 5905)	1
  (0, 2136)	1
  (0, 6080)	1
  (0, 4309)	1
  (0, 4786)	1
  (0, 390)	1
  (0, 2137)	1
  (0, 2135)	1
  (0, 4006)	1
  (0, 5750)	1
  (0, 3252)	1
  (0, 3151)	1
  (0, 5862)	1
  (0, 404)	1


In [72]:
tfidf_vectoriser = TfidfVectorizer()
X_train_tfidf = tfidf_vectoriser.fit_transform(X_train)
X_test_tfidf = tfidf_vectoriser.transform(X_test)

In [73]:
model_bow = MultinomialNB()
model_bow.fit(X_train_bow, y_train)
 
model_tfidf = MultinomialNB()
model_tfidf.fit(X_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 6540)","[[0. ,0. ,0.24,...,0.19,0. ,0.35], [2.29,4.93,0. ,...,0. ,0.22,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 6540)","[[-9.69,-9.69,-9.47,...,-9.52,-9.69,-9.39], [-7.9 ,-7.31,-9.09,...,-9.09,-8.88,-9.09]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,6540


In [74]:
def evaluate(name, model, X_test_vec, y_test):
    print(name)
    y_pred = model.predict(X_test_vec)
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
    print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
    print("\nFull report:")
    print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
    return y_pred


In [75]:
evaluate("nb + tfidf", model_tfidf, X_test_tfidf, y_test)

nb + tfidf
Accuracy : 0.9632
Precision: 1.0000
Recall   : 0.7248
F1-score : 0.8405

Full report:
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



array([0, 0, 0, ..., 0, 0, 0], shape=(1115,))

In [76]:
evaluate("nb + bow", model_bow, X_test_bow, y_test)

nb + bow
Accuracy : 0.9848
Precision: 0.9648
Recall   : 0.9195
F1-score : 0.9416

Full report:
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.98      0.98      0.98      1115



array([0, 0, 0, ..., 0, 0, 0], shape=(1115,))